# CS406 – Nhận diện biển số xe Việt Nam (VNLP)

Notebook chính: **dữ liệu → CRNN → PARSeq → so sánh → YOLO → pipeline end-to-end**.

* Mọi cấu hình nằm trong **`config.yaml`** ở gốc repo (đường dẫn, siêu tham số, bật/tắt từng phần).
* Muốn đổi nhanh mà không sửa file: điền `OVERRIDES` ở cell dưới, vd `{"notebook.quick_test": False, "crnn.epochs": 50}`.
* Kết quả lưu ở `outputs/<experiment>/` (kèm `config_used.yaml`, `run_info.json`) – đổi `experiment` trong config mỗi lần thử nghiệm.

| Môi trường | Cần chuẩn bị |
|---|---|
| Local | Kernel `.venv` của repo; đã chạy `python -m vnlp.data.split_recognition` + `python -m vnlp.data.split_detection` |
| Kaggle | Từ máy local: `python -m vnlp.kaggle_push` – tự upload code + data, chạy notebook (GPU), in link theo dõi |

> `notebook.quick_test: true` → mỗi split chỉ vài ảnh, 1 epoch: để kiểm tra code trên máy không có GPU.

## 0. Ghi đè cấu hình (tùy chọn)

In [ ]:
OVERRIDES = {
    # "notebook.quick_test": False,
    # "notebook.run_detection": True,
    # "crnn.epochs": 50,
}

# Chỉ dùng trên Kaggle khi KHÔNG có dataset code (cs406-vnlp-code) -> clone từ GitHub
REPO_URL = "https://github.com/Khang210105/CS406_VNLP_2027.git"
REPO_BRANCH = "main"
KAGGLE_DATA_DIR = None   # None = tự tìm thư mục chứa splits/ trong /kaggle/input

## 1. Môi trường & cấu hình

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    REPO_ROOT = Path("/kaggle/working/CS406_VNLP_2027")
    code_hits = sorted(Path("/kaggle/input").glob("**/vnlp/config.py"))  # dataset do vnlp.kaggle_push upload
    if not REPO_ROOT.exists() and code_hits:
        shutil.copytree(code_hits[0].parents[1], REPO_ROOT)  # copy sang working để ghi được (yolo weights...)
    if not REPO_ROOT.exists():
        url = REPO_URL
        try:  # repo private -> thêm secret GITHUB_TOKEN (Add-ons > Secrets)
            from kaggle_secrets import UserSecretsClient
            url = url.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
        except Exception:
            pass
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_BRANCH, url, str(REPO_ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_ROOT / "requirements-kaggle.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "strhub @ git+https://github.com/baudm/parseq.git@1902db043c029a7e03a3818c616c06600af574be"], check=True)

    data_dir = Path(KAGGLE_DATA_DIR) if KAGGLE_DATA_DIR else None
    if data_dir is None:
        hits = sorted(Path("/kaggle/input").glob("**/splits*/recognition"))
        if not hits:
            raise FileNotFoundError("Không tìm thấy splits*/recognition trong /kaggle/input – đã Add Input dataset chưa?")
        data_dir = hits[0].parents[1]
    os.environ["VNLP_DATA_DIR"] = str(data_dir)
    os.environ["VNLP_OUTPUT_DIR"] = "/kaggle/working/outputs"
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config.yaml").exists():
        if REPO_ROOT.parent == REPO_ROOT:
            raise FileNotFoundError("Không tìm thấy gốc repo (config.yaml) – hãy mở notebook bên trong repo")
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from vnlp import config
config.reload(OVERRIDES)
NB = config.CFG["notebook"]

print("Experiment:", config.EXPERIMENT, "| commit:", config._git_commit())
print("Kaggle :", IS_KAGGLE, "| Device:", config.get_device())
print("Repo   :", REPO_ROOT)
print("Data   :", config.DATA_DIR)
print("Output :", config.OUTPUT_DIR)
print("Notebook flags:", NB)

In [ ]:
import shutil
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image as IPImage, display

from vnlp.utils.io import imread, load_samples, view_of
from vnlp.recognition.preprocess import four_point_transform, increase_resolution, to_gray, deskew, enhance_image, preprocess_plate

def show_png(path, width=900):
    path = Path(path)
    display(IPImage(filename=str(path), width=width)) if path.exists() else print("(chưa có)", path)

def show_samples(samples, ncols=6, title=None, size=2.2):
    if not samples:
        print("(không có mẫu)"); return
    nrows = (len(samples) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size * 1.4, nrows * size), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (img_path, caption) in zip(axes.flat, samples):
        img = imread(img_path)
        if img is not None:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(caption, fontsize=9)
    if title:
        fig.suptitle(title, fontweight="bold")
    plt.tight_layout(); plt.show()

### Data dùng cho lần chạy này
`quick_test` copy vài ảnh mỗi split (rải đều các view) sang `outputs/quick_data` – code train y hệt lúc chạy thật.

In [ ]:
QUICK = NB["quick_test"]
# Chạy thật: lưu đúng đường dẫn mặc định trong config (demo, label_plates dùng lại được).
# quick_test: tách riêng ra outputs/quick để không đè kết quả thật.
RUN_OUT = config.OUTPUT_DIR / "quick" if QUICK else config.OUTPUT_DIR
CRNN_OUT = RUN_OUT / "recognition" / "crnn" if QUICK else config.CRNN_OUTPUT_DIR
PARSEQ_OUT = RUN_OUT / "recognition" / "parseq" if QUICK else config.PARSEQ_OUTPUT_DIR
DET_OUT = RUN_OUT / "detection" if QUICK else config.DET_OUTPUT_DIR
RECOG_DIRS = {"train": config.RECOG_TRAIN_DIR, "val": config.RECOG_VAL_DIR, "test": config.RECOG_TEST_DIR}
DET_YAML = config.DET_DATA_YAML

def spread_by_view(paths, n, key):
    by_view = {}
    for p in sorted(paths):
        by_view.setdefault(key(p), []).append(p)
    picked, i = [], 0
    while len(picked) < n and any(i < len(v) for v in by_view.values()):
        picked += [v[i] for v in by_view.values() if i < len(v)]
        i += 1
    return picked[:n]

if QUICK:
    n = NB["quick_n"]
    quick_root = config.OUTPUT_DIR / "quick_data"
    shutil.rmtree(quick_root, ignore_errors=True)
    for split, src in RECOG_DIRS.items():
        for d in spread_by_view([p for p in Path(src).iterdir() if p.is_dir()], n, lambda p: view_of(p.name)):
            shutil.copytree(d, quick_root / "recognition" / split / d.name)
    RECOG_DIRS = {k: quick_root / "recognition" / k for k in RECOG_DIRS}

    if config.DET_DATA_YAML.exists():
        det_q = quick_root / "detection"
        for split in ("train", "val", "test"):
            imgs = spread_by_view(list((config.DET_SPLIT_DIR / "images" / split).iterdir()), n, lambda p: p.stem.split("_")[0])
            for sub in ("images", "labels"):
                (det_q / sub / split).mkdir(parents=True, exist_ok=True)
            for img in imgs:
                shutil.copy2(img, det_q / "images" / split / img.name)
                shutil.copy2(config.DET_SPLIT_DIR / "labels" / split / f"{img.stem}.txt", det_q / "labels" / split)
        shutil.copy2(config.DET_DATA_YAML, det_q / "data.yaml")
        DET_YAML = det_q / "data.yaml"

    config.reload({**OVERRIDES, "crnn.epochs": 1, "parseq.epochs": 1, "detection.train.epochs": 1,
                   "crnn.batch_size": 4, "parseq.batch_size": 4, "detection.train.batch": 2,
                   "detection.train.imgsz": 320, "detection.eval.imgsz": 320, "detection.eval.batch": 2})

RUN_INFO = config.snapshot(RUN_OUT)  # config_used.yaml + run_info.json: biết kết quả này từ code/tham số nào
for k, v in RECOG_DIRS.items():
    print(f"recognition/{k:<5} {len(load_samples(v)):>5} mẫu  <- {v}")
print("detection data.yaml:", DET_YAML, "(có)" if Path(DET_YAML).exists() else "(CHƯA CÓ)")
print("Kết quả lưu tại   :", RUN_OUT)

## 2. Tổng quan dữ liệu

In [ ]:
if NB["run_data_overview"]:
    rec = pd.read_csv(config.RECOG_MANIFEST)
    print("Recognition (số crop):")
    display(rec.pivot_table(index="view", columns="split", values="sample_id", aggfunc="count",
                            margins=True, margins_name="ALL")[["train", "val", "test", "ALL"]])
    leak = set(rec.query("split=='train'").plate) & set(rec.query("split!='train'").plate)
    print("Số biển khác nhau:", rec.groupby("split")["plate"].nunique().to_dict(), "| biển train trùng val/test:", len(leak))

    if config.DET_MANIFEST.exists():
        det = pd.read_csv(config.DET_MANIFEST)
        print("\nDetection (số ảnh):")
        display(det.pivot_table(index="view", columns="split", values="image_id", aggfunc="count",
                                margins=True, margins_name="ALL")[["train", "val", "test", "ALL"]])
        print("Nguồn quyết định split:", det["split_source"].value_counts().to_dict())

    rec.assign(n=rec["plate"].str.len()).groupby(["n", "split"]).size().unstack(fill_value=0) \
       .plot.bar(figsize=(7, 3), title="Phân bố độ dài biển số")
    plt.tight_layout(); plt.show()

In [ ]:
if NB["run_data_overview"]:
    by_view = {}
    for s in load_samples(RECOG_DIRS["train"]):
        by_view.setdefault(view_of(s[2]), []).append(s)
    show_samples([(p, f"{sid}\n{t}") for v in sorted(by_view) for p, t, sid in by_view[v][:6]],
                 ncols=6, title="Crop biển số tập train (mỗi hàng 1 view)")

    if config.DET_MANIFEST.exists():
        det_imgs = spread_by_view(list((Path(DET_YAML).parent / "images" / "train").iterdir()), 5, lambda p: p.stem.split("_")[0])
        fig, axes = plt.subplots(1, len(det_imgs), figsize=(4 * len(det_imgs), 3.2), squeeze=False)
        for ax, p in zip(axes.flat, det_imgs):
            img = cv2.cvtColor(imread(p), cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
            for line in (Path(DET_YAML).parent / "labels" / "train" / f"{p.stem}.txt").read_text().splitlines():
                _, cx, cy, bw, bh = map(float, line.split())
                ax.add_patch(plt.Rectangle(((cx - bw / 2) * w, (cy - bh / 2) * h), bw * w, bh * h, fill=False, color="lime", lw=2))
            ax.imshow(img); ax.set_title(p.stem, fontsize=9); ax.axis("off")
        fig.suptitle("Ảnh detection + box nhãn (mỗi view 1 ảnh)", fontweight="bold"); plt.tight_layout(); plt.show()

### Các bước tiền xử lý (`recognition/preprocess.py`)

In [ ]:
if NB["run_data_overview"]:
    demo = [s for v in sorted(by_view) for s in by_view[v][:1]]
    steps = ["gốc", "perspective", "upscale", "gray", "deskew", "enhance", "final"]
    fig, axes = plt.subplots(len(demo), len(steps), figsize=(len(steps) * 2.6, len(demo) * 1.4), squeeze=False)
    for row, (path, text, sid) in enumerate(demo):
        img = imread(path)
        persp = four_point_transform(img)
        up = increase_resolution(persp, config.IMG_W, config.IMG_H)
        gray = to_gray(up); desk = deskew(gray); enh = enhance_image(desk)
        for col, (name, im) in enumerate(zip(steps, [img, persp, up, gray, desk, enh, preprocess_plate(img)])):
            ax = axes[row, col]
            ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if im.ndim == 3 else im, cmap="gray"); ax.axis("off")
            if row == 0:
                ax.set_title(name, fontsize=9)
        axes[row, 0].text(-5, 0, f"{sid}\n{text}", fontsize=8, ha="right", va="top")
    plt.tight_layout(); plt.show()

## 3. CRNN + CTC

In [ ]:
from vnlp.recognition import crnn

if NB["run_crnn"]:
    crnn.train(RECOG_DIRS["train"], RECOG_DIRS["val"], CRNN_OUT)
    show_png(CRNN_OUT / "ocr_training_results.png")
    crnn_pred, crnn_metrics = crnn.evaluate(RECOG_DIRS["test"], CRNN_OUT / "ocr_model.pth", CRNN_OUT, verbose=False)
    display(crnn_metrics)

## 4. PARSeq

In [ ]:
from vnlp.recognition import parseq

if NB["run_parseq"]:
    parseq_ckpt = parseq.train(RECOG_DIRS["train"], RECOG_DIRS["val"], PARSEQ_OUT)
    show_png(PARSEQ_OUT / "parseq_training_results.png")
    parseq_pred, parseq_metrics = parseq.evaluate(RECOG_DIRS["test"], parseq_ckpt, PARSEQ_OUT, verbose=False)
    display(parseq_metrics)

## 5. So sánh OCR & phân tích lỗi
Cột **+postprocess**: áp dụng `vn_plate.postprocess_ocr` (sửa nhầm chữ/số theo luật biển VN).

In [ ]:
from vnlp.recognition.postprocess import postprocess_ocr
from vnlp.utils.io import normalize_label
from vnlp.utils.metrics import summarize_predictions

def with_postprocess(df_pred):
    recs = [{"sample_id": r.sample_id, "gt": r.gt,
             "pred": normalize_label(postprocess_ocr(r.pred, return_two_lines=False)["formatted"])}
            for r in df_pred.itertuples()]
    return summarize_predictions(recs)[1]

results = {}
if NB["run_crnn"]:
    results["CRNN"], results["CRNN +post"] = crnn_metrics, with_postprocess(crnn_pred)
if NB["run_parseq"]:
    results["PARSeq"], results["PARSeq +post"] = parseq_metrics, with_postprocess(parseq_pred)
if results:
    compare = pd.concat({k: v.set_index("view")[["plate_acc", "cer"]] for k, v in results.items()}, axis=1)
    display(compare)
    compare.to_csv(RUN_OUT / "recognition" / "ocr_comparison.csv")

In [ ]:
def error_gallery(df_pred, name, n=18):
    wrong = df_pred[~df_pred["correct"]].head(n)
    print(f"{name}: {(~df_pred['correct']).sum()}/{len(df_pred)} biển sai")
    show_samples([(Path(RECOG_DIRS["test"]) / r.sample_id / "image.jpg", f"GT {r.gt}\nPred {r.pred}") for r in wrong.itertuples()],
                 ncols=6, title=f"{name} – các biển đọc sai")

if NB["run_crnn"]:
    error_gallery(crnn_pred, "CRNN")
if NB["run_parseq"]:
    error_gallery(parseq_pred, "PARSeq")

## 6. Detection – YOLO

In [ ]:
from vnlp.detection import train as det_train, evaluate as det_eval

YOLO_BEST = DET_OUT / config.DET_RUN_NAME / "weights" / "best.pt"
if NB["run_detection"]:
    if not Path(DET_YAML).exists():
        print("⚠️ Chưa có", DET_YAML, "-> chạy python -m vnlp.data.split_detection")
    else:
        YOLO_BEST = det_train.train(DET_YAML, DET_OUT)
        show_png(DET_OUT / config.DET_RUN_NAME / "results.png")
        display(det_eval.evaluate(YOLO_BEST, DET_YAML, DET_OUT))

## 7. Pipeline end-to-end: ảnh gốc → YOLO → crop → OCR → `vn_plate`
Đánh giá trên **tập test detection** (cùng biển với tập test OCR, không rò rỉ).
Một biển GT tính đúng nếu có dự đoán trùng khớp hoàn toàn.

In [ ]:
from vnlp.pipeline import PlatePipeline, evaluate_end_to_end, ground_truth_by_image

if NB["run_pipeline"]:
    ocr_name = NB["pipeline_ocr"]
    ocr_weights = (PARSEQ_OUT / "parseq_model.ckpt") if ocr_name == "parseq" else (CRNN_OUT / "ocr_model.pth")
    pipe = PlatePipeline(yolo_weights=YOLO_BEST, ocr=ocr_name, ocr_weights=ocr_weights)

    gt_test = ground_truth_by_image("test")
    ids = spread_by_view(list(gt_test), NB["quick_n"], lambda k: k.split("_")[0]) if QUICK else None
    e2e_images, e2e_summary = evaluate_end_to_end(pipe, "test", image_ids=ids, out_dir=RUN_OUT / "pipeline")
    display(e2e_summary)

    shown = [(k, gt_test[k]) for k in spread_by_view(list(gt_test), NB["pipeline_show"], lambda k: k.split("_")[0])]
    fig, axes = plt.subplots((len(shown) + 2) // 3, 3, figsize=(15, 4 * ((len(shown) + 2) // 3)), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (image_id, (path, gt)) in zip(axes.flat, shown):
        frame = imread(path)
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        for p in pipe.predict(frame):
            x1, y1, x2, y2 = p["box"]
            ok = p["text"] in gt
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="lime" if ok else "red", lw=2))
            ax.text(x1, y1 - 4, p["text"], color="black", backgroundcolor="lime" if ok else "red", fontsize=9)
        ax.set_title(f"{image_id} | GT: {', '.join(gt)}", fontsize=9)
    plt.tight_layout(); plt.show()

## 8. Kết quả
Mọi file (weights, history, biểu đồ, predictions CSV) nằm trong `RUN_OUT`.
Trên Kaggle: tab **Output** để tải về.

In [ ]:
for p in sorted(RUN_OUT.rglob("*")):
    if p.is_file() and not ({"lightning_logs", "test_eval", "quick", "quick_data"} & set(p.relative_to(RUN_OUT).parts[:-1])) and p.name != ".gitkeep":
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(RUN_OUT)}")